In [1]:
import os
from pathlib import Path
import joblib
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score
)
import argparse

In [ ]:
from plotly.subplots import make_subplots
import plotly.express as px

In [2]:
ambiente = 'project'

In [3]:
PORCENTAJE_SAMPLE_DATA = 0.7
EVERY_N_YEARS = 2
RANDOM_SEED = 0

SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_power_kW_m'
]

GMM_FEATURES = [
    'wind_speed_ms', 
    'wave_energy', 'wave_height_m', 'wave_period_s', 'wave_power_kW_m'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_energy', 'wave_period_s']
SORT_FEATURES = ["wave_energy", "wave_period_s", "wind_speed_ms"]
N_CLUSTERS = 6
EXTREME_THRESHOLD = 0.9

RF_FEATURES = [
    'wind_speed_ms', 
    'wave_energy', 'wave_height_m', 'wave_period_s', 'wave_power_kW_m'
]
TARGET = "gmm_sea_state_level"
TEST_SIZE = 0.25
N_ESTIMATORS = 500
MAX_DEPTH = None
MIN_SAMPLES_LEAF = 10
CLASS_WEIGHT = "balanced"
N_JOBS = -1

In [ ]:
scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'

In [4]:
coast_names = (
    spark.sql(
        f"""
            SELECT DISTINCT coast_name
            FROM cor_{ambiente}.silver.swell_metrics
        """
    )
).toPandas()['coast_name'].tolist()

In [5]:
coast = coast_names[0]

In [6]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE coast_name = '{coast}'
                AND YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

In [7]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}
data_sample = (
        data
        .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
        .drop('coast_year_month')
    ).toPandas()

In [ ]:
scaler = joblib.load(scaler_path.format(coast))
scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)

In [ ]:
gmm = GaussianMixture(
        n_components=N_CLUSTERS,
        covariance_type="full",
        random_state=RANDOM_SEED
    )
data_sample["gmm_cluster"] = gmm.fit_predict(scaled_df[GMM_FEATURES])
data_sample["gmm_cluster_probability"] = gmm.predict_proba(scaled_df[GMM_FEATURES]).max(axis=1)

In [ ]:
cluster_summary = (
        data_sample.groupby("gmm_cluster")[GMM_FEATURES]
        .mean()
        .sort_values(SORT_FEATURES)
    )
cluster_order = {
        old_cluster: new_cluster + 1
        for new_cluster, old_cluster in enumerate(cluster_summary.index)
    }

In [ ]:
data_sample["gmm_sea_state_level"] = data_sample["gmm_cluster"].map(cluster_order)

data_sample['gmm_mask_extremo'] = data_sample[EXTREME_FEATURES[0]] > data_sample[EXTREME_FEATURES[0]].quantile(EXTREME_THRESHOLD)
for feature in EXTREME_FEATURES[1:]:
    data_sample['gmm_mask_extremo'] &= data_sample[feature] > data_sample[feature].quantile(EXTREME_THRESHOLD)

data_sample.loc[data_sample['gmm_mask_extremo'], 'gmm_sea_state_level'] = 7

In [ ]:
q_5 = data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.05)
q_10 = data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.10)
q_15 = data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.15)
if not (
        all(data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.05)>=0.5) and
        all(data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.10)>=0.7) and
        all(data_sample.groupby('gmm_sea_state_level')['gmm_cluster_probability'].quantile(0.15)>=0.8) 
    ):
        print(f'El modelo GMM para la costa {coast} no es lo suficientemente bueno, saltando...')
else:
        print(f'El modelo GMM para la costa {coast} si es lo suficientemente bueno')

print(f'Quantiles: 0.05: {q_5}, 0.10: {q_10}, 0.15: {q_15}')

In [ ]:
fig = make_subplots(
    rows=len(EXTREME_FEATURES),
    cols=2,
    specs=[
        [{"type": "scene", "rowspan": len(EXTREME_FEATURES)}, {"type": "xy"}],
        *[
            [None, {"type": "xy"}]
            for _ in range(len(EXTREME_FEATURES) - 1)
        ]
    ],
    subplot_titles=[
        f'Muestra datos de la costa {coast}',
        *[f'Outliers por {feature}' for feature in EXTREME_FEATURES]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    column_widths=[1/3, 2/3]
)

# Gráfica 3D
fig_3d = px.scatter_3d(
    data_sample,
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_energy',
    color='gmm_sea_state',
    color_discrete_map={
        False: 'blue',
        True: 'red'
    }
)

fig_3d.update_traces(marker=dict(size=3))

for trace in fig_3d.data:
    fig.add_trace(trace, row=1, col=1)

# Gráficas 2D
for i, feature in enumerate(EXTREME_FEATURES):
    tmp_fig = px.scatter(
        data_sample,
        x='datetime',
        y=feature,
        color='gmm_sea_state',
        color_discrete_map={
            False: 'blue',
            True: 'red'
        }
    )

    tmp_fig.update_traces(marker=dict(size=3))

    for trace in tmp_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=i+1, col=2)

    threshold = data_sample[feature].quantile(EXTREME_THRESHOLD)

    # Línea horizontal
    fig.add_shape(
        type="line",
        x0=data_sample['datetime'].min(),
        x1=data_sample['datetime'].max(),
        y0=threshold,
        y1=threshold,
        line=dict(
            color="black",
            width=2,
            dash="dash"
        ),
        row=i+1,
        col=2
    )

fig.update_layout(
    height=len(EXTREME_FEATURES)*250,
    width=1300,
    title=f'Análisis de outliers - Costa {coast}',
    uirevision='constant',
    showlegend=False,
    margin=dict(t=90),
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Energía de la ola (J)',
        aspectmode='cube'
    )
)

pct_outlier = (data_sample['gmm_sea_state'].value_counts(normalize=True)*100).loc['Mar extremo']
fig.add_annotation(
    text=f"pct es outlier: {pct_outlier:.2f}%",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.1,
    showarrow=False,
    font=dict(size=14)
)

fig.show()